<a href="https://colab.research.google.com/github/rouuuuuuu/PFA/blob/main/distilbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install torch
!pip install scikit-learn
!pip install transformers datasets
!pip install -U transformers
!pip install --upgrade transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
!pip install -U transformers
!pip install datasets


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Charger le CSV sans colonnes
df = pd.read_csv('/content/df_small_balanced_9000.csv', header=None, engine='python', on_bad_lines='skip')

# Assigner la première ligne comme noms de colonnes
df.columns = df.iloc[0]
df = df.drop(0).reset_index(drop=True)

# Encodage des labels (Positive=0, Neutral=1, Negative=2)
label_map = {'Positive': 0, 'Neutral': 1, 'Negative': 2}
df['label'] = df['Sentiment'].map(label_map)

# Séparer train/test
train_data, val_data = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

# Convertir en Dataset Hugging Face
train_dataset = Dataset.from_pandas(train_data.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_data.reset_index(drop=True))

# Tokenizer
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    return tokenizer(examples['CommentText'], truncation=True, padding=True)

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

# Charger le modèle
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/7199 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import DataCollatorWithPadding
from transformers import Trainer, TrainingArguments, EvalPrediction
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=30,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="epoch",  # Log at each epoch
    metric_for_best_model="f1",  # Criteria for selecting the best model
    logging_steps=10,
    report_to="none",  # or "tensorboard" if you want to visualize logs
)

# Fonction de calcul des métriques
def compute_metrics(pred: EvalPrediction):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, average='weighted'),
        'recall': recall_score(labels, preds, average='weighted'),
        'f1': f1_score(labels, preds, average='weighted')
    }

# Initialiser et entraîner le modèle
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

trainer.train()

# Évaluer le modèle après l'entraînement
eval_result = trainer.evaluate()

# Afficher les métriques
print("Résultats de l'évaluation finale :")
for key, value in eval_result.items():
    print(f"{key}: {value:.4f}")


Step,Training Loss
450,0.615700
900,0.417800
1350,0.270700
1800,0.172500
2250,0.115500
2700,0.072000
3150,0.050600
3600,0.033100
4050,0.033700
4500,0.027200


In [ ]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
from transformers import Trainer, TrainingArguments

# 9. Définir les arguments d'entraînement
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=30,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)




# 10. Fonction de calcul des métriques
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, average='weighted'),
        'recall': recall_score(labels, preds, average='weighted'),
        'f1': f1_score(labels, preds, average='weighted')
    }

# 11. Initialiser et entraîner le modèle
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator

)

trainer.train()
# Évaluer le modèle après l'entraînement
eval_result = trainer.evaluate()

# Afficher les métriques
print("Résultats de l'évaluation :")
for key, value in eval_result.items():
    print(f"{key}: {value:.4f}")


In [ ]:

# 17. Évaluer le modèle
eval_result = trainer.evaluate()
print("Résultat de l'évaluation : ", eval_result)

# 18. Prédictions sur de nouveaux commentaires
sample_comment = "This video was very informative and helpful."
inputs = tokenizer(sample_comment, return_tensors="pt", truncation=True, padding=True)
logits = model(**inputs).logits
predicted_class = logits.argmax(-1).item()

# Afficher le sentiment prédit
sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
print("Sentiment prédit pour le commentaire : ", sentiment_map[predicted_class])

In [ ]:
# Sauvegarder le modèle et le tokenizer
trainer.save_model("./my_model")           # Sauve le modèle entraîné
tokenizer.save_pretrained("./my_model")     # Sauve aussi le tokenizer


('./my_model/tokenizer_config.json',
 './my_model/special_tokens_map.json',
 './my_model/vocab.txt',
 './my_model/added_tokens.json',
 './my_model/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Charger modèle + tokenizer
model = AutoModelForSequenceClassification.from_pretrained("./my_model")
tokenizer = AutoTokenizer.from_pretrained("./my_model")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Charger le modèle et le tokenizer sauvegardés
model = AutoModelForSequenceClassification.from_pretrained("./my_model")
tokenizer = AutoTokenizer.from_pretrained("./my_model")

# Fonction de prédiction
def predict_sentiment(comment):
    # Tokenisation du commentaire
    inputs = tokenizer(comment, return_tensors="pt", truncation=True, padding=True)

    # Si le modèle est sur GPU, déplace les tenseurs aussi sur le GPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Prédiction
    with torch.no_grad():
        logits = model(**inputs).logits

    # Obtenir la classe prédite (0 = Negative, 1 = Neutral, 2 = Positive)
    predicted_class = logits.argmax(-1).item()

    # Mapper le label à son nom
    sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return sentiment_map[predicted_class]

# Test de la fonction
sample_comment = "This video was very informative and helpful."
predicted_sentiment = predict_sentiment(sample_comment)
print(f"Sentiment prédit pour le commentaire : {predicted_sentiment}")


Sentiment prédit pour le commentaire : Positive
